In [ ]:
import random
from typing import 

# Chrome version hiện tại
CHROME_MAJOR = 131  # Update theo Chrome release
CHROME_FULL = "131.0.6778.139"

# Windows versions
WINDOWS_VERSIONS = [
    ("Windows NT 10.0; Win64; x64", "Windows", False),
    ("Windows NT 10.0; WOW64", "Windows", False),
]

# macOS versions
MAC_VERSIONS = [
    ("Macintosh; Intel Mac OS X 10_15_7", "macOS", False),
    ("Macintosh; Intel Mac OS X 13_5", "macOS", False),
    ("Macintosh; Intel Mac OS X 14_2", "macOS", False),
]

# Linux
LINUX_VERSIONS = [
    ("X11; Linux x86_64", "Linux", False),
]

# Android devices (device_string, android_version, model_name)
ANDROID_DEVICES = [
    ("Linux; Android 14; SM-S918B", "14", "SM-S918B"),      # S23 Ultra
    ("Linux; Android 14; SM-S921B", "14", "SM-S921B"),      # S24
    ("Linux; Android 15; Pixel 9 Pro", "15", "Pixel 9 Pro"),
    ("Linux; Android 14; Pixel 8", "14", "Pixel 8"),
    ("Linux; Android 14; Redmi Note 13", "14", "Redmi Note 13"),
]

# iOS devices
IOS_DEVICES = [
    ("iPhone; CPU iPhone OS 17_5 like Mac OS X", "17.5"),
    ("iPhone; CPU iPhone OS 18_0 like Mac OS X", "18.0"),
]

def generate_chrome_version() -> str:
    """Generate realistic Chrome version."""
    build = random.randint(6700, 6999)
    patch = random.randint(0, 200)
    return f"{CHROME_MAJOR}.0.{build}.{patch}"

def generate_sec_ch_ua(chrome_version: str) -> str:
    """
    Generate sec-ch-ua header.
    Example: "Chromium";v="131", "Google Chrome";v="131", "Not/A)Brand";v="99"
    """
    major = chrome_version.split('.')[0]
    # Random "Not Brand" để tránh fingerprint giống hệt nhau
    not_brands = [
        f'"Not/A)Brand";v="99"',
        f'"Not_A Brand";v="8"',
        f'"Not:A-Brand";v="99"',
    ]
    return f'"Chromium";v="{major}", "Google Chrome";v="{major}", {random.choice(not_brands)}'

def generate_desktop_fingerprint() -> dict:
    """Generate complete desktop browser fingerprint."""
    
    # Random OS
    os_data = random.choice(WINDOWS_VERSIONS + MAC_VERSIONS + LINUX_VERSIONS)
    platform_string, platform_name, is_mobile = os_data
    
    # Chrome version
    chrome_ver = generate_chrome_version()
    
    # User-Agent
    user_agent = (
        f"Mozilla/5.0 ({platform_string}) AppleWebKit/537.36 "
        f"(KHTML, like Gecko) Chrome/{chrome_ver} Safari/537.36"
    )
    
    # Viewport theo OS
    if "Windows" in platform_string:
        viewport = random.choice([[1920, 1080], [1366, 768], [2560, 1440]])
    elif "Macintosh" in platform_string:
        viewport = random.choice([[1440, 900], [1920, 1080], [2560, 1600]])
    else:  # Linux
        viewport = [1920, 1080]
    
    return {
        # User-Agent
        "user_agent": user_agent,
        
        # Client Hints Headers (Chrome 89+)
        "sec_ch_ua": generate_sec_ch_ua(chrome_ver),
        "sec_ch_ua_mobile": "?0",  # Desktop = ?0
        "sec_ch_ua_platform": f'"{platform_name}"',
        
        # Navigator properties
        "platform": "Win32" if platform_name == "Windows" else ("MacIntel" if platform_name == "macOS" else "Linux x86_64"),
        "mobile": False,
        
        # Viewport
        "viewport": {"width": viewport[0], "height": viewport[1]},
        "screen": {"width": viewport[0], "height": viewport[1]},
        
        # Language & Timezone
        "language": random.choice(["en-US", "en-GB", "vi-VN"]),
        "timezone": random.choice([
            "America/New_York",
            "Europe/London",
            "Asia/Ho_Chi_Minh"
        ]),
        
        # Hardware
        "hardware_concurrency": random.choice([4, 8, 12, 16]),
        "device_memory": random.choice([4, 8, 16, 32]),
    }

def generate_mobile_fingerprint() -> dict:
    """Generate complete mobile browser fingerprint."""
    
    # Random device
    if random.random() < 0.8:  # 80% Android, 20% iOS
        # Android
        device_data = random.choice(ANDROID_DEVICES)
        device_string, android_ver, model = device_data
        
        chrome_ver = generate_chrome_version()
        
        user_agent = (
            f"Mozilla/5.0 ({device_string}) AppleWebKit/537.36 "
            f"(KHTML, like Gecko) Chrome/{chrome_ver} Mobile Safari/537.36"
        )
        
        platform_name = "Android"
        viewport = random.choice([
            [393, 851],   # Pixel
            [412, 915],   # Samsung
            [360, 800],   # Standard
        ])
        
        return {
            "user_agent": user_agent,
            "sec_ch_ua": generate_sec_ch_ua(chrome_ver),
            "sec_ch_ua_mobile": "?1",  # Mobile = ?1
            "sec_ch_ua_platform": '"Android"',
            "sec_ch_ua_model": f'"{model}"',  # Android có thêm model
            
            "platform": "Linux armv8l",
            "mobile": True,
            "viewport": {"width": viewport[0], "height": viewport[1]},
            "screen": {"width": viewport[0], "height": viewport[1]},
            "language": random.choice(["en-US", "vi-VN"]),
            "timezone": "Asia/Ho_Chi_Minh",
            "hardware_concurrency": random.choice([4, 6, 8]),
            "device_memory": random.choice([4, 6, 8]),
        }
    else:
        # iOS
        device_data = random.choice(IOS_DEVICES)
        device_string, ios_ver = device_data
        
        user_agent = (
            f"Mozilla/5.0 ({device_string}) AppleWebKit/605.1.15 "
            f"(KHTML, like Gecko) Version/{ios_ver} Mobile/15E148 Safari/604.1"
        )
        
        viewport = random.choice([
            [390, 844],   # iPhone 13/14
            [393, 852],   # iPhone 15
        ])
        
        # iOS Safari KHÔNG có sec-ch-ua (chỉ Chrome mới có)
        return {
            "user_agent": user_agent,
            # iOS Safari không gửi Client Hints
            "sec_ch_ua": None,
            "sec_ch_ua_mobile": None,
            "sec_ch_ua_platform": None,
            
            "platform": "iPhone",
            "mobile": True,
            "viewport": {"width": viewport[0], "height": viewport[1]},
            "screen": {"width": viewport[0], "height": viewport[1]},
            "language": random.choice(["en-US", "vi-VN"]),
            "timezone": "Asia/Ho_Chi_Minh",
            "hardware_concurrency": random.choice([4, 6, 8]),
            "device_memory": random.choice([4, 6, 8]),
        }

def get_request_headers(fingerprint: dict) -> dict[str, str]:
    """
    Convert fingerprint to HTTP request headers.
    Dùng cho requests/httpx.
    """
    headers = {
        "User-Agent": fingerprint["user_agent"],
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": f"{fingerprint['language']},en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    # Add Client Hints nếu có (Chrome on Android/Desktop)
    if fingerprint.get("sec_ch_ua"):
        headers["sec-ch-ua"] = fingerprint["sec_ch_ua"]
        headers["sec-ch-ua-mobile"] = fingerprint["sec_ch_ua_mobile"]
        headers["sec-ch-ua-platform"] = fingerprint["sec_ch_ua_platform"]
        
        # Android có thêm model
        if fingerprint.get("sec_ch_ua_model"):
            headers["sec-ch-ua-model"] = fingerprint["sec_ch_ua_model"]
    
    return headers

# ==========================================
# PLAYWRIGHT INTEGRATION
# ==========================================

def apply_fingerprint_to_playwright(page, fingerprint: Dict):
    """
    Apply fingerprint to Playwright page.
    Call AFTER page creation, BEFORE navigation.
    """
    
    # Override navigator properties
    script = f"""
        Object.defineProperty(navigator, 'platform', {{
            get: () => '{fingerprint["platform"]}'
        }});
        
        Object.defineProperty(navigator, 'hardwareConcurrency', {{
            get: () => {fingerprint["hardware_concurrency"]}
        }});
        
        Object.defineProperty(navigator, 'deviceMemory', {{
            get: () => {fingerprint["device_memory"]}
        }});
        
        // Remove webdriver flag
        Object.defineProperty(navigator, 'webdriver', {{
            get: () => false
        }});
        
        // Override languages
        Object.defineProperty(navigator, 'languages', {{
            get: () => ['{fingerprint["language"]}', 'en']
        }});
    """
    
    # Add Client Hints nếu có
    if fingerprint.get("sec_ch_ua"):
        script += f"""
        Object.defineProperty(navigator, 'userAgentData', {{
            get: () => ({{
                brands: [
                    {{brand: "Chromium", version: "{CHROME_MAJOR}"}},
                    {{brand: "Google Chrome", version: "{CHROME_MAJOR}"}},
                    {{brand: "Not/A)Brand", version: "99"}}
                ],
                mobile: {str(fingerprint["mobile"]).lower()},
                platform: {fingerprint["sec_ch_ua_platform"]}
            }})
        }});
        """
    
    page.add_init_script(script)

# ==========================================
# USAGE EXAMPLES
# ==========================================

if __name__ == "__main__":
    print("=== DESKTOP FINGERPRINT ===")
    desktop = generate_desktop_fingerprint()
    print(f"User-Agent: {desktop['user_agent']}")
    print(f"sec-ch-ua: {desktop['sec_ch_ua']}")
    print(f"sec-ch-ua-mobile: {desktop['sec_ch_ua_mobile']}")
    print(f"sec-ch-ua-platform: {desktop['sec_ch_ua_platform']}")
    print()
    
    print("=== MOBILE ANDROID FINGERPRINT ===")
    mobile = generate_mobile_fingerprint()
    print(f"User-Agent: {mobile['user_agent']}")
    print(f"sec-ch-ua: {mobile['sec_ch_ua']}")
    print(f"sec-ch-ua-mobile: {mobile['sec_ch_ua_mobile']}")
    print(f"sec-ch-ua-platform: {mobile['sec_ch_ua_platform']}")
    if mobile.get('sec_ch_ua_model'):
        print(f"sec-ch-ua-model: {mobile['sec_ch_ua_model']}")
    print()
    
    print("=== REQUEST HEADERS ===")
    headers = get_request_headers(desktop)
    for key, value in headers.items():
        print(f"{key}: {value}")
    print()
    
    # Example: Using with requests
    print("=== REQUESTS EXAMPLE ===")
    print("""
import requests

fingerprint = generate_desktop_fingerprint()
headers = get_request_headers(fingerprint)

response = requests.get('https://httpbin.org/headers', headers=headers)
print(response.json())
    """)
    
    # Example: Using with Playwright
    print("=== PLAYWRIGHT EXAMPLE ===")
    print("""
from playwright.sync_api import sync_playwright

fingerprint = generate_desktop_fingerprint()

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context(
        user_agent=fingerprint['user_agent'],
        viewport=fingerprint['viewport'],
        screen=fingerprint['screen'],
        locale=fingerprint['language'],
        timezone_id=fingerprint['timezone'],
        is_mobile=fingerprint['mobile'],
    )
    
    page = context.new_page()
    
    # Apply fingerprint BEFORE navigation
    apply_fingerprint_to_playwright(page, fingerprint)
    
    # Set extra HTTP headers (Client Hints)
    if fingerprint.get('sec_ch_ua'):
        context.set_extra_http_headers({
            'sec-ch-ua': fingerprint['sec_ch_ua'],
            'sec-ch-ua-mobile': fingerprint['sec_ch_ua_mobile'],
            'sec-ch-ua-platform': fingerprint['sec_ch_ua_platform'],
        })
    
    page.goto('https://bot.sannysoft.com/')
    input("Press Enter to close...")
    """)


=== DESKTOP FINGERPRINT ===
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 13_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6955.142 Safari/537.36
sec-ch-ua: "Chromium";v="131", "Google Chrome";v="131", "Not/A)Brand";v="99"
sec-ch-ua-mobile: ?0
sec-ch-ua-platform: "macOS"

=== MOBILE ANDROID FINGERPRINT ===
User-Agent: Mozilla/5.0 (iPhone; CPU iPhone OS 17_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.5 Mobile/15E148 Safari/604.1
sec-ch-ua: None
sec-ch-ua-mobile: None
sec-ch-ua-platform: None

=== REQUEST HEADERS ===
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 13_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6955.142 Safari/537.36
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8
Accept-Language: en-US,en;q=0.9
Accept-Encoding: gzip, deflate, br
DNT: 1
Connection: keep-alive
Upgrade-Insecure-Requests: 1
sec-ch-ua: "Chromium";v="131", "Google Chrome";v="131", "Not/A)Brand";v="99"
s

In [9]:

import requests

fingerprint = generate_desktop_fingerprint()
headers = get_request_headers(fingerprint)

response = requests.get('https://httpbin.org/headers', headers=headers)
print(response.json())
    


{'headers': {'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8', 'Accept-Encoding': 'gzip, deflate, br', 'Accept-Language': 'en-US,en;q=0.9', 'Dnt': '1', 'Host': 'httpbin.org', 'Sec-Ch-Ua': '"Chromium";v="131", "Google Chrome";v="131", "Not_A Brand";v="8"', 'Sec-Ch-Ua-Mobile': '?0', 'Sec-Ch-Ua-Platform': '"Windows"', 'Upgrade-Insecure-Requests': '1', 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6955.37 Safari/537.36', 'X-Amzn-Trace-Id': 'Root=1-6a06c8de-5266cb2f3f81ed6b7135d5c2'}}


In [37]:
generate_desktop_fingerprint()

{'user_agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6986.104 Safari/537.36',
 'sec_ch_ua': '"Chromium";v="131", "Google Chrome";v="131", "Not/A)Brand";v="99"',
 'sec_ch_ua_mobile': '?0',
 'sec_ch_ua_platform': '"Linux"',
 'platform': 'Linux x86_64',
 'mobile': False,
 'viewport': {'width': 1920, 'height': 1080},
 'screen': {'width': 1920, 'height': 1080},
 'language': 'en-GB',
 'timezone': 'Asia/Ho_Chi_Minh',
 'hardware_concurrency': 16,
 'device_memory': 4}

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [40]:
fingerprint = generate_desktop_fingerprint()
fingerprint

{'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6864.1 Safari/537.36',
 'sec_ch_ua': '"Chromium";v="131", "Google Chrome";v="131", "Not/A)Brand";v="99"',
 'sec_ch_ua_mobile': '?0',
 'sec_ch_ua_platform': '"macOS"',
 'platform': 'MacIntel',
 'mobile': False,
 'viewport': {'width': 1920, 'height': 1080},
 'screen': {'width': 1920, 'height': 1080},
 'language': 'en-US',
 'timezone': 'Asia/Ho_Chi_Minh',
 'hardware_concurrency': 4,
 'device_memory': 8}

In [52]:
fingerprint = generate_desktop_fingerprint()
fingerprint

{'user_agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6803.69 Safari/537.36',
 'sec_ch_ua': '"Chromium";v="131", "Google Chrome";v="131", "Not_A Brand";v="8"',
 'sec_ch_ua_mobile': '?0',
 'sec_ch_ua_platform': '"Linux"',
 'platform': 'Linux x86_64',
 'mobile': False,
 'viewport': {'width': 1920, 'height': 1080},
 'screen': {'width': 1920, 'height': 1080},
 'language': 'en-GB',
 'timezone': 'America/New_York',
 'hardware_concurrency': 16,
 'device_memory': 16}